### Importing Packages

In [66]:
from langchain_community.retrievers import WikipediaRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain.vectorstores import Chroma
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_groq import ChatGroq

### Data Source Retrievers - Wikipedia Retriever

In [15]:
retriever = WikipediaRetriever(top_k_results=2, lang='en')

In [16]:
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

docs = retriever.invoke(query)

In [18]:
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  


--- Result 1 ---
Content:
The Dominion of Pakistan, officially Pakistan, was an independent federal dominion in the British Commonwealth of Nations, which existed from 14 August 1947 to 23 March 1956. It was created by the passing of the Indian Independence Act 1947 by the British parliament, which also created an independent Dominion of India.
The new dominion consisted of those presidencies and provinces of British India which were allocated to it in the Partition of India. Until 1947, these regions had been ruled by the United Kingdom as a part of the British Empire.
Its status as a federal dominion within the British Empire ended in 1956 with the completion of the Constitution of Pakistan, which established the country as a republic. The constitution also administratively split the nation into West Pakistan and East Pakistan. Until then, these provinces had been governed as a singular entity, despite being separate geographic exclaves. Eventually, the East became Bangladesh and th

### Data Source Retrievers - VectorStore Retriever

In [19]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [24]:
vector_store = Chroma.from_documents(
    embedding=HuggingFaceEndpointEmbeddings(
        model="BAAI/bge-large-en-v1.5",
    ),
    documents=documents,
    collection_name='sample'
)

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [26]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [27]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
OpenAI provides powerful embedding models.


In [28]:
results = vector_store.similarity_search(query, k=2)

In [29]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
OpenAI provides powerful embedding models.


### Search-Strategy Retrievers - MMA

In [31]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [35]:
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=HuggingFaceEndpointEmbeddings(
        model="BAAI/bge-large-en-v1.5",
    ),
)

In [36]:
retriever = vector_store.as_retriever(
    search_type="mmr",                   
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

In [37]:
query = "What is langchain?"
results = retriever.invoke(query)

In [38]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 3 ---
Embeddings are vector representations of text.


### Search-Strategy Retrievers - Multi query

In [39]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [40]:
vector_store = FAISS.from_documents(
    documents=all_docs,
    embedding=HuggingFaceEndpointEmbeddings(
        model="BAAI/bge-large-en-v1.5",
    ),
)

In [55]:
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [56]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
    llm=ChatGroq(model='llama-3.1-8b-instant')
)

In [57]:
query = "How to improve energy levels and maintain balance?"

In [58]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [59]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
******************************************************************************************************************************************************

--- Result 1 ---
Python balances readability with power, making it a popular system design language.

--- Result 2 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 3 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.

--- Result 6 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.


### Search-Strategy - Contextual Compression Retriever

In [60]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [61]:
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=HuggingFaceEndpointEmbeddings(
        model="BAAI/bge-large-en-v1.5",
    ),
)

In [62]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [67]:
llm = ChatGroq(model='llama-3.1-8b-instant')
compressor = LLMChainExtractor.from_llm(llm)

In [68]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [69]:
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [70]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.
        Photosynthesis

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.

--- Result 3 ---
Photosynthesis does not occur in animal cells.
